# FNEWS — Text modeling

**Research objective.** Saved TF-IDF + LinearSVC and frozen pretrained-text baseline evidence, with configuration and validation results.

This consolidated notebook reuses saved, executed research evidence. It does not train models or perform new held-out-test selection. `SOURCE_TRUE` and `SOURCE_NON_TRUE` are Fakeddit source-taxonomy labels, not independent real-world fact-checking verdicts.

In [ ]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd()
if not (ROOT / "data/research_dataset").exists():
    ROOT = Path("/Users/aka_ank/Desktop/FNEWS")
DATA = ROOT / "data/research_dataset"
REPORTS = ROOT / "reports"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13, "axes.labelsize": 11})

def paired(split):
    frame = pd.read_parquet(DATA / "manifests" / f"{split}_manifest.parquet")
    return frame.loc[(frame.source_dataset == "Fakeddit") & (frame.download_status == "verified_reused_or_processed")].copy().reset_index(drop=True)

def canonical_source_labels(raw):
    """Fakeddit raw 1=true; return 0=SOURCE_TRUE, 1=SOURCE_NON_TRUE."""
    return 1 - np.asarray(raw, dtype=int)

def source_names(values):
    return np.where(np.asarray(values) == 0, "SOURCE_TRUE", "SOURCE_NON_TRUE")

def show_frame(frame, rows=10):
    display(frame.head(rows))


## Text baselines and evaluation

_Source preserved from `notebooks/archive/2026-10-27/research/04_text_model_training.ipynb`._

## TF-IDF + LinearSVC

LinearSVC learns a separating margin. Its decision function is not a calibrated probability. The stored classifier was fit with raw Fakeddit labels, where class 1 means source-true; the application translates this at presentation time.

In [2]:
metrics=json.loads((REPORTS/'experiments/research_dataset_baselines/metrics.json').read_text());display(pd.DataFrame(metrics['results']['tfidf_svm']['validation']['per_class']).T);print('Saved validation accuracy:',metrics['results']['tfidf_svm']['validation']['accuracy']);print('Saved validation macro-F1:',metrics['results']['tfidf_svm']['validation']['macro_f1'])
pipe=joblib.load(REPORTS/'experiments/research_dataset_baselines/tfidf_svm/model.joblib');print(pipe)

,precision,recall,f1-score,support
true,0.810526,0.692446,0.746848,556.000000
fake,0.626638,0.761273,0.687425,377.000000
accuracy,0.720257,0.720257,0.720257,0.720257
macro avg,0.718582,0.726860,0.717136,933.000000
weighted avg,0.736222,0.720257,0.722837,933.000000


Saved validation accuracy: 0.7202572347266881
Saved validation macro-F1: 0.7171364351800763


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=50000, min_df=2,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('svm', LinearSVC())])


In [3]:
pred=pd.read_parquet(REPORTS/'experiments/research_dataset_baselines/tfidf_svm/validation_predictions.parquet');print('Stored prediction records:',len(pred));display(pred.head())
bert=pd.read_parquet(REPORTS/'experiments/research_dataset_baselines/pretrained_text_frozen/validation_predictions.parquet');print('Frozen pretrained-text records:',len(bert))

Stored prediction records: 933


,sample_id,label,prediction
0,fakeddit:validate:11nlcg,0,1
1,fakeddit:validate:11upsa,0,1
2,fakeddit:validate:1661by,0,0
3,fakeddit:validate:1bwjj3,0,0
4,fakeddit:validate:1ee6zy,0,0


Frozen pretrained-text records: 933


## Training methodology and limitations

The SVM has no epoch-level history; plotting a learning curve or loss curve would be fabricated. Its regularization and feature configuration are recovered from the saved pipeline. The frozen pretrained-text baseline provides a comparison but does not establish domain-general factuality performance.

## Reproducibility and limitations

The original notebooks remain unchanged under `notebooks/archive/2026-10-27/`. Rebuild this active suite with `python scripts/consolidate_notebooks.py`. Archived outputs are historical records, not claims of a new execution.